# Week 2: Classification, telecom churn
Goal: find customers who are likely to leave (`Churn = Yes`).
This notebook is reused tomorrow, so it is kept tidy.

## 1. Look first

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Rows and columns:", df.shape)
print()
print(df["Churn"].value_counts())
print("Churn rate:", round((df["Churn"] == "Yes").mean() * 100, 1), "%")

Rows and columns: (7043, 21)

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn rate: 26.5 %


**Write:** 1,869 of 7,043 customers churned, which is 26.5%, so about 1 in 4 customers left.

### Checking TotalCharges 
It should be a number, but pandas may read it as text.

In [2]:
print("dtype of TotalCharges:", df["TotalCharges"].dtype)

# Blank strings (spaces) are not real numbers. Convert, and bad values become NaN.
total_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")
bad = df[total_numeric.isna()]
print("Rows that are not numbers:", len(bad))
print("Their tenure values:", bad["tenure"].unique())

df["TotalCharges"] = total_numeric

dtype of TotalCharges: str
Rows that are not numbers: 11
Their tenure values: [0]


**Write:** TotalCharges was read as text because 11 rows hold a blank space; all 11 have tenure 0 (brand new customers), so they have no charges yet. I turned them into NaN and fill them with the training median later.

## 2. Split (stratified, so both sets keep the same churn rate)

In [3]:
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])   # customerID is just a label, not useful

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Train:", len(X_train), "| Test:", len(X_test))
print("Churn rate train:", round(y_train.mean(), 3), "| test:", round(y_test.mean(), 3))

Train: 5634 | Test: 1409
Churn rate train: 0.265 | test: 0.265


**Write:** The split gives 5,634 training rows and 1,409 test rows, and both keep a churn rate of about 0.265 thanks to stratify.

## 3. Baseline: predict "no churn" for everyone

In [4]:
base_pred = np.zeros(len(y_test), dtype=int)
print("Baseline accuracy:", round(accuracy_score(y_test, base_pred), 3))
print("Baseline recall  :", round(recall_score(y_test, base_pred), 3))

Baseline accuracy: 0.735
Baseline recall  : 0.0


**Write:** Saying "no churn" for everyone gets accuracy 0.735 but recall 0.0, so it looks fine and catches nobody who leaves.

## 4. LogisticRegression(max_iter=1000)
Number columns: fill missing with median, then scale. Text columns: one hot encode.

In [5]:
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

prep = ColumnTransformer([
    ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
clf = Pipeline([("prep", prep), ("lr", LogisticRegression(max_iter=1000))])
clf.fit(X_train, y_train)            # fitted on TRAIN only

pred = clf.predict(X_test)
print("Accuracy :", round(accuracy_score(y_test, pred), 3))
print("Precision:", round(precision_score(y_test, pred), 3))
print("Recall   :", round(recall_score(y_test, pred), 3))
print("Confusion matrix (rows = real, columns = predicted; order No, Yes):")
print(confusion_matrix(y_test, pred))

Accuracy : 0.806
Precision: 0.657
Recall   : 0.559
Confusion matrix (rows = real, columns = predicted; order No, Yes):
[[926 109]
 [165 209]]


**Write:** Accuracy is 0.806, precision 0.657 and recall 0.559. The matrix shows 209 churners caught and 165 missed, with 109 loyal customers wrongly flagged.

## 5. Precision or recall? Then threshold 0.3

**Which matters more: recall.** Money view (these numbers are my assumptions, change them if the company gives real ones):
- Losing a customer costs about 12 months of bill, about **$900** (churners pay about $74 a month).
- A retention offer costs about **$50** per customer we contact.
- A missed churner costs $900. A wrongly flagged loyal customer costs only $50. So a miss is about 18 times worse than a false alarm, and we should catch more churners even if some alerts are wrong.

In [6]:
proba = clf.predict_proba(X_test)[:, 1]      # chance of churn for each customer
pred_03 = (proba >= 0.3).astype(int)

print("Threshold 0.3")
print("Precision:", round(precision_score(y_test, pred_03), 3))
print("Recall   :", round(recall_score(y_test, pred_03), 3))
print("Accuracy :", round(accuracy_score(y_test, pred_03), 3))
print(confusion_matrix(y_test, pred_03))

# Simple money check with the assumptions above
value_saved, offer_cost, success_rate = 900, 50, 0.30   # 30% of caught churners accept the offer
def profit(p):
    tp = ((p == 1) & (y_test == 1)).sum()
    contacted = (p == 1).sum()
    return tp * success_rate * value_saved - contacted * offer_cost
print()
print("Estimated profit at 0.5 threshold: $", round(profit(pred)))
print("Estimated profit at 0.3 threshold: $", round(profit(pred_03)))

Threshold 0.3
Precision: 0.519
Recall   : 0.754
Accuracy : 0.749
[[774 261]
 [ 92 282]]

Estimated profit at 0.5 threshold: $ 40530
Estimated profit at 0.3 threshold: $ 48990


**Write:** At threshold 0.3 recall rises from 0.559 to 0.754 (282 churners caught instead of 209) while precision drops from 0.657 to 0.519. With my money assumptions this is the better choice because the extra customers saved are worth more than the extra offers.

## 6. So what (three lines for a manager)

1. Our model finds about 3 in 4 customers who will leave (recall 0.75), but only about half of the people it flags really would leave (precision 0.52).
2. Send a retention offer to every customer with a churn chance of 30% or more, because a missed customer costs about $900 and an offer costs about $50.
3. Check the real offer cost and acceptance rate, then pick the final threshold again with those numbers.